## **Fuzzy Rule-Based Classifier**

A fuzzy rule uses **linguistic terms** with **gradual membership** instead of sharp thresholds: `IF mean concave points is high AND mean radius is medium THEN malignant`. A case belongs to *high* with a degree between 0 and 1, so a small change in the measurement changes the decision smoothly. This is the bridge between the **expert system** ( human-readable rules ) and **learning from data**.

This notebook implements a classifier in the style of **Ishibuchi and Nojima** ( fuzzy rules learned from data, one winner rule ):

1. each attribute is divided into **fuzzy sets** ( *low / medium / high* ) with triangular membership functions placed on quantiles of the training set,
2. **every combination** of up to 2 ( or 3 ) attribute-terms is a candidate antecedent; its **class** and its **rule weight** come from the **fuzzy support and fuzzy confidence** on the training set,
3. a new case is classified by the **winner rule**: the rule with the highest `membership × weight`.

**Notebooks of this series:** 1. CN2 Rule Induction · 2. Expert Systems · 3. Case-Based Reasoning · 4. Apriori for Classification ( CBA ) · 5. Simple Rule Learners ( ZeroR, OneR, PRISM ) · 6. RIPPER · 7. Rules from Decision Trees · 8. RuleFit · 9. Fuzzy Rule-Based Classifier

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

In [3]:
#Helper Functions: Evaluation Reports ( Confusion Matrix, Metrics, ROC and Precision-Recall )
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
                             confusion_matrix, roc_curve, precision_recall_curve)

def plot_confusion(cm, labels, ax, title="Confusion matrix"):
    ax.imshow(cm, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45 if len(labels) > 4 else 0, ha="right" if len(labels) > 4 else "center")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=10 if len(labels) > 4 else 13,
                    color="white" if cm[i, j] > cm.max() / 2 else DARK)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

def report_binary(y_true, y_pred, y_score, name="model", labels=("negative", "positive"), plot=True):
    """Confusion matrix, ROC and Precision-Recall curves + a Series with the main metrics (positive class = 1)."""
    y_true, y_pred, y_score = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_score)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred),
               "F1": f1_score(y_true, y_pred), "MCC": matthews_corrcoef(y_true, y_pred),
               "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score)}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        plot_confusion(confusion_matrix(y_true, y_pred), list(labels), axes[0], f"Confusion matrix – {name}")
        fpr, tpr, _ = roc_curve(y_true, y_score)
        axes[1].plot(fpr, tpr, color=RED, lw=2.4, label=f"AUC = {metrics['ROC AUC']:.3f}"); axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--")
        axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
        p, r, _ = precision_recall_curve(y_true, y_score)
        axes[2].plot(r, p, color=DARK, lw=2.4, label=f"AP = {metrics['PR AUC']:.3f}"); axes[2].axhline(y_true.mean(), color=GRAY, ls="--")
        axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title("Precision–Recall curve"); axes[2].legend(loc="lower left")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)

def report_multiclass(y_true, y_pred, y_proba, labels, name="model", plot=True):
    """Confusion matrix, one-vs-rest ROC curves and per-class F1 + a Series with the main metrics."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = len(labels)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
               "macro recall": recall_score(y_true, y_pred, average="macro"), "macro F1": f1_score(y_true, y_pred, average="macro"),
               "MCC": matthews_corrcoef(y_true, y_pred), "ROC AUC (ovr)": roc_auc_score(y_true, y_proba, multi_class="ovr"),
               "log loss": log_loss(y_true, y_proba, labels=list(range(k)))}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
        plot_confusion(confusion_matrix(y_true, y_pred, labels=list(range(k))), list(labels), axes[0], f"Confusion matrix – {name}")
        cmap = plt.get_cmap("tab10")
        for c in range(k):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), y_proba[:, c])
            axes[1].plot(fpr, tpr, color=cmap(c), lw=1.6, label=f"{labels[c]}")
        axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
        axes[1].set_title(f"ROC one-vs-rest (macro AUC = {metrics['ROC AUC (ovr)']:.3f})"); axes[1].legend(fontsize=7, ncol=2 if k > 5 else 1, loc="lower right")
        f1s = f1_score(y_true, y_pred, average=None)
        axes[2].bar(range(k), f1s, color=RED, width=0.65); axes[2].set_xticks(range(k)); axes[2].set_xticklabels(labels, rotation=45 if k > 4 else 0, ha="right" if k > 4 else "center")
        axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("F1"); axes[2].set_title("F1 per class")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)
def annotate_heatmap(ax, mat, fmt="{:.2f}"):
    """Write the values on an imshow heat map with a readable text color."""
    mat = np.asarray(mat, dtype=float)
    lo, hi = np.nanmin(mat), np.nanmax(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, "–" if np.isnan(v) else fmt.format(v), ha="center", va="center", color="white" if (not np.isnan(v) and v > (lo + hi) / 2) else DARK)

### **1. Dataset and Preparation**

The same problem and the same **70/30 stratified split** of the other notebooks of the series: **malignant ( 1 ) × benign ( 0 )** tumours, the 10 *mean* attributes of the Breast Cancer dataset.

In [4]:
#Load the Breast Cancer Dataset ( Positive Class = Malignant )
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
cols = [c for c in data.data.columns if c.startswith("mean ")]            # 10 "mean" attributes
X = data.data[cols]
y = (data.target == 0).astype(int)                                        # 1 = malignant, 0 = benign
CLASS_NAMES = ["benign", "malignant"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape, "| malignant in train:", f"{y_train.mean():.1%}")
X_train.head(3)

train: (398, 10) | test: (171, 10) | malignant in train: 37.2%


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension
78,20.18,23.97,143.70,1245.0,0.129,0.345,0.375,0.160,0.291,0.081
330,16.03,15.51,105.80,793.2,0.095,0.137,0.120,0.070,0.178,0.060
378,13.66,15.15,88.27,580.6,0.083,0.075,0.042,0.025,0.179,0.059


In [ ]:
#Dictionary Used to Collect the Test Metrics of This Notebook ( the Cells Below Add Their Results to It )
results = {}

### **2. Fuzzy Partitions**

With $m$ terms per attribute, the membership functions are triangles centred on equally spaced **quantiles** of the training data ( with *shoulders* at both ends: the first term is 1 below the first centre, the last term is 1 above the last centre ). The fuzzy measures of a rule $A \Rightarrow c$ on the $n$ training cases are:

$$\text{support} = \frac{1}{n} \sum_{x \in c} \mu_A(x) \qquad \text{confidence} = \frac{\sum_{x \in c} \mu_A(x)}{\sum_{x} \mu_A(x)} \qquad \text{weight } CF = 2 \cdot \text{confidence} - 1$$

where $\mu_A(x)$ is the product of the memberships of $x$ in the terms of the antecedent.

In [ ]:
#Fuzzy Partition of Every Attribute ( Triangular Membership Functions on Quantiles ) and the Membership Matrix
from itertools import combinations

TERMS = {3: ["low", "medium", "high"], 5: ["very low", "low", "medium", "high", "very high"]}

def make_partition(X, n_terms):
    return {a: np.quantile(X[a], np.linspace(0.05, 0.95, n_terms)) for a in X.columns}                   # centres of the triangles

def membership(values, centres):
    mu = np.zeros((len(values), len(centres)))
    v = np.asarray(values, dtype=float)
    for i, c in enumerate(centres):
        left = centres[i - 1] if i > 0 else None
        right = centres[i + 1] if i < len(centres) - 1 else None
        up = np.ones(len(v)) if left is None else np.clip((v - left) / (c - left), 0, 1)                    # rising side ( or shoulder )
        down = np.ones(len(v)) if right is None else np.clip((right - v) / (right - c), 0, 1)               # falling side ( or shoulder )
        mu[:, i] = np.where(v <= c, up, down)
        if left is None:
            mu[v <= c, i] = 1.0
        if right is None:
            mu[v >= c, i] = 1.0
    return mu

def membership_matrix(X, partition, n_terms):
    cols, names = [], []
    for a in X.columns:
        cols.append(membership(X[a].to_numpy(), partition[a]))
        names += [(a, t) for t in TERMS[n_terms]]
    return np.hstack(cols), names

partition3 = make_partition(X_train, 3)
M3, term_names = membership_matrix(X_train, partition3, 3)
print("membership matrix:", M3.shape, "( 10 attributes x 3 terms ) | memberships of one case in 'mean radius':", M3[0, :3].round(2), "( the sum of the memberships of an attribute is 1 )")
grid = np.linspace(X_train["mean concave points"].min(), X_train["mean concave points"].max(), 300)
mu_grid = membership(grid, partition3["mean concave points"])
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for k, color in enumerate([DARK, GRAY, RED]):
    ax.plot(grid, mu_grid[:, k], color=color, lw=2.4, label=TERMS[3][k])
ax.set_xlabel("mean concave points"); ax.set_ylabel("membership"); ax.set_title("Fuzzy sets of one attribute"); ax.legend()
plt.tight_layout(); plt.show()

### **3. Learning the Rules and Classifying with the Winner Rule**

Rules with a fuzzy support below `min_support` or a weight below `min_cf` are discarded. For prediction, the score of a class is the **highest `membership × weight`** among the rules of that class; if no rule fires, the training prevalence is used.

In [ ]:
#Fuzzy Rule Learning ( Antecedents with up to max_len Terms ) and Winner-Rule Classification
class FuzzyRules:
    def __init__(self, n_terms=3, max_len=2, min_support=0.02, min_cf=0.3, top_k=None):
        self.n_terms, self.max_len, self.min_support, self.min_cf, self.top_k = n_terms, max_len, min_support, min_cf, top_k

    def _antecedents(self, M):
        n_attr = M.shape[1] // self.n_terms
        out = []
        for L in range(1, self.max_len + 1):
            for attrs in combinations(range(n_attr), L):
                for terms in np.ndindex(*([self.n_terms] * L)):
                    out.append(tuple(a * self.n_terms + t for a, t in zip(attrs, terms)))
        return out

    def _mu(self, M, ante):
        return M[:, list(ante)].prod(axis=1)

    def fit(self, X, y):
        y = np.asarray(y)
        self.partition_ = make_partition(X, self.n_terms)
        M, self.names_ = membership_matrix(X, self.partition_, self.n_terms)
        n, rules = len(y), []
        self.prior_ = y.mean()
        self.n_candidates_ = 0
        for ante in self._antecedents(M):
            mu = self._mu(M, ante)
            total = mu.sum()
            self.n_candidates_ += 1
            if total == 0:
                continue
            conf1 = mu[y == 1].sum() / total
            cls = int(conf1 >= 0.5)
            conf = conf1 if cls == 1 else 1 - conf1
            support = mu[y == cls].sum() / n
            cf = 2 * conf - 1
            if support >= self.min_support and cf >= self.min_cf:
                rules.append({"ante": ante, "cls": cls, "support": support, "confidence": conf, "cf": cf})
        rules.sort(key=lambda r: (-r["cf"] * r["support"]))
        self.all_rules_ = rules
        self.rules_ = rules[:self.top_k] if self.top_k else rules
        return self

    def scores(self, X):
        M, _ = membership_matrix(X, self.partition_, self.n_terms)
        s = np.zeros((len(X), 2))
        for r in self.rules_:
            s[:, r["cls"]] = np.maximum(s[:, r["cls"]], self._mu(M, r["ante"]) * r["cf"])
        return s

    def predict_proba(self, X):
        s = self.scores(X)
        fired = s.sum(axis=1) > 0
        p1 = np.full(len(X), self.prior_)
        p1[fired] = (s[fired, 1] + 1e-3) / (s[fired].sum(axis=1) + 2e-3)
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def n_rules(self):
        return len(self.rules_)

    def rule_text(self, r):
        return " AND ".join(f"{self.names_[i][0].replace('mean ', '')} is {self.names_[i][1]}" for i in r["ante"])

    def rules_table(self, k=10):
        return pd.DataFrame([{"IF": self.rule_text(r), "THEN": CLASS_NAMES[r["cls"]], "fuzzy support": r["support"], "fuzzy confidence": r["confidence"], "weight ( CF )": r["cf"]} for r in self.rules_[:k]])

fz = FuzzyRules(n_terms=3, max_len=2).fit(X_train, y_train)
proba_fz = fz.predict_proba(X_test)[:, 1]
pred_fz = fz.predict(X_test)
print(f"candidate antecedents: {fz.n_candidates_} | rules kept ( support ≥ 2%, weight ≥ 0.3 ): {fz.n_rules()} ( malignant: {sum(r['cls'] == 1 for r in fz.rules_)}, benign: {sum(r['cls'] == 0 for r in fz.rules_)} )")
results["Fuzzy rules"] = report_binary(y_test, pred_fz, proba_fz, "Fuzzy rule-based classifier", CLASS_NAMES)
pd.set_option("display.max_colwidth", 90)
fz.rules_table(8).round(3)

### **4. Quality Metrics Specific to Fuzzy Rules**

In [ ]:
#How Many Rules Are Needed? Test Metrics Using Only the k Best Rules ( Ranked by Weight x Support )
rows = []
for k in [1, 2, 5, 10, 20, 50, 100, len(fz.all_rules_)]:
    m = FuzzyRules(n_terms=3, max_len=2, top_k=k).fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    covered = (m.scores(X_test).sum(axis=1) > 0).mean()
    rows.append({"rules used": min(k, len(m.all_rules_)), "test accuracy": accuracy_score(y_test, (p > 0.5).astype(int)), "test ROC AUC": roc_auc_score(y_test, p), "test cases with at least one firing rule": covered})
topk = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogx(topk["rules used"], topk["test accuracy"], color=RED, marker="o", label="test accuracy"); ax.semilogx(topk["rules used"], topk["test ROC AUC"], color=DARK, marker="s", label="test ROC AUC")
ax.set_xlabel("rules used ( log scale )"); ax.set_ylim(0.5, 1.02); ax.legend(); ax.set_title("Test performance vs number of rules used")
plt.tight_layout(); plt.show()
topk.round(3)

In [ ]:
#Fuzzy vs Crisp Partitions and the Effect of the Number of Terms and of the Rule Length ( Repeated CV on the Training Set )
from sklearn.model_selection import RepeatedStratifiedKFold
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

class CrispRules(FuzzyRules):
    # same rules, but each case belongs 100% to its nearest term ( membership 0 or 1 ): a classical discretisation
    def scores(self, X):
        M, _ = membership_matrix(X, self.partition_, self.n_terms)
        M = (M == M.reshape(len(X), -1, self.n_terms).max(axis=2).repeat(self.n_terms, axis=1)).astype(float)
        s = np.zeros((len(X), 2))
        for r in self.rules_:
            s[:, r["cls"]] = np.maximum(s[:, r["cls"]], self._mu(M, r["ante"]) * r["cf"])
        return s

    def fit(self, X, y):
        y = np.asarray(y)
        self.partition_ = make_partition(X, self.n_terms)
        M, self.names_ = membership_matrix(X, self.partition_, self.n_terms)
        M = (M == M.reshape(len(X), -1, self.n_terms).max(axis=2).repeat(self.n_terms, axis=1)).astype(float)
        self.prior_, rules = y.mean(), []
        for ante in self._antecedents(M):
            mu = self._mu(M, ante); total = mu.sum()
            if total == 0:
                continue
            conf1 = mu[y == 1].sum() / total
            cls = int(conf1 >= 0.5); conf = conf1 if cls == 1 else 1 - conf1
            support, cf = mu[y == cls].sum() / len(y), 2 * conf - 1
            if support >= self.min_support and cf >= self.min_cf:
                rules.append({"ante": ante, "cls": cls, "support": support, "confidence": conf, "cf": cf})
        self.all_rules_ = self.rules_ = sorted(rules, key=lambda r: (-r["cf"] * r["support"]))
        return self

def cv_model(factory):
    accs, aucs, sizes = [], [], []
    for a, b_ in cv_rep.split(X_train, y_train):
        m = factory().fit(X_train.iloc[a], y_train.iloc[a])
        p = m.predict_proba(X_train.iloc[b_])[:, 1]
        accs.append(((p > 0.5).astype(int) == y_train.iloc[b_].to_numpy()).mean()); aucs.append(roc_auc_score(y_train.iloc[b_], p)); sizes.append(m.n_rules())
    return {"accuracy ( mean )": np.mean(accs), "accuracy ( std )": np.std(accs), "ROC AUC ( mean )": np.mean(aucs), "ROC AUC ( std )": np.std(aucs), "rules ( mean )": np.mean(sizes)}

variants = {"fuzzy, 3 terms, up to 2 conditions": lambda: FuzzyRules(3, 2), "crisp, 3 terms, up to 2 conditions": lambda: CrispRules(3, 2),
            "fuzzy, 3 terms, up to 1 condition": lambda: FuzzyRules(3, 1), "fuzzy, 3 terms, up to 3 conditions": lambda: FuzzyRules(3, 3),
            "fuzzy, 5 terms, up to 2 conditions": lambda: FuzzyRules(5, 2)}
cv_variants = pd.DataFrame({name: cv_model(f) for name, f in variants.items()}).T
cv_variants.round(3)

In [ ]:
#Reading One Decision: Which Rules Fire for a Test Case ( the Winner Rule Decides )
pos = int(np.flatnonzero((pred_fz == y_test.to_numpy()) & (proba_fz > 0.8))[0])
M_case, _ = membership_matrix(X_test.iloc[[pos]], fz.partition_, 3)
fired_rules = sorted([(fz._mu(M_case, r["ante"])[0] * r["cf"], r) for r in fz.rules_], key=lambda t: -t[0])[:4]
print(f"test case {pos}: true class = {CLASS_NAMES[y_test.iloc[pos]]}, P(malignant) = {proba_fz[pos]:.2f}")
for strength, r in fired_rules:
    print(f"  strength {strength:.2f} = membership x weight {r['cf']:.2f} | IF {fz.rule_text(r)} THEN {CLASS_NAMES[r['cls']]}")

### **5. Evaluation**

In [ ]:
#Reference Models on the Same Split ( Is the Technique Better than a Trivial or a Standard Model? )
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

references = {"Majority class": DummyClassifier(strategy="prior"),
              "Decision tree ( depth 3 )": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
              "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))}
reference_rows = {}
for ref_name, model in references.items():
    model.fit(X_train, y_train)
    reference_rows[ref_name] = report_binary(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1], ref_name, CLASS_NAMES, plot=False)
metric_cols = ["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]
pd.concat([pd.DataFrame(results).T[metric_cols], pd.DataFrame(reference_rows).T[metric_cols]]).round(3)

In [ ]:
#Repeated CV of the Reference Models and the Fuzzy Classifier on the Same Folds
from sklearn.model_selection import cross_validate
cv_rows = {"Fuzzy rules ( 3 terms, 2 conditions )": cv_variants.loc["fuzzy, 3 terms, up to 2 conditions", ["accuracy ( mean )", "accuracy ( std )", "ROC AUC ( mean )", "ROC AUC ( std )"]].to_dict()}
for ref_name, model in references.items():
    res = cross_validate(model, X_train, y_train, cv=cv_rep, scoring=["accuracy", "roc_auc"])
    cv_rows[ref_name] = {"accuracy ( mean )": res["test_accuracy"].mean(), "accuracy ( std )": res["test_accuracy"].std(), "ROC AUC ( mean )": res["test_roc_auc"].mean(), "ROC AUC ( std )": res["test_roc_auc"].std()}
pd.DataFrame(cv_rows).T.round(3)

In [ ]:
#Uncertainty of the Test Metrics ( Bootstrap of the Test Cases, 1,000 Resamples )
def bootstrap_ci(y_true, pred, score, n_boot=1000, seed=RANDOM_STATE):
    rng_b = np.random.default_rng(seed)
    y_true, pred, score = np.asarray(y_true), np.asarray(pred), np.asarray(score)
    acc, auc = [], []
    for _ in range(n_boot):
        i = rng_b.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue
        acc.append((pred[i] == y_true[i]).mean())
        auc.append(roc_auc_score(y_true[i], score[i]))
    return pd.DataFrame({"accuracy": np.percentile(acc, [2.5, 50, 97.5]), "ROC AUC": np.percentile(auc, [2.5, 50, 97.5])}, index=["2.5%", "median", "97.5%"]).T

print("the test set has", len(y_test), "cases: differences smaller than these intervals are not reliable")
bootstrap_ci(y_test, pred_fz, proba_fz).round(3)

In [ ]:
#Calibration of the Predicted Probabilities ( Brier Score and Reliability Curve )
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

proba_fz_ARRAY = np.asarray(proba_fz, dtype=float)
frac_pos, mean_pred = calibration_curve(y_test, proba_fz_ARRAY, n_bins=5, strategy="quantile")                       # bins with the same number of cases
print(f"Brier score: {brier_score_loss(y_test, proba_fz_ARRAY):.4f} | Brier score of a model that always predicts the training prevalence: {brier_score_loss(y_test, np.full(len(y_test), y_train.mean())):.4f} ( lower is better )")
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], color=GRAY, ls="--", label="perfect calibration"); ax.plot(mean_pred, frac_pos, color=RED, marker="o", label="Fuzzy rules")
ax.set_xlabel("mean predicted probability ( 5 bins of equal size )"); ax.set_ylabel("observed share of malignant"); ax.set_title("Reliability curve"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### **6. Takeaways**

* **Gradual membership:** the decision changes smoothly with the measurement. In the cross-validation the fuzzy and the crisp partitions have the **same accuracy** ( 0.902 and 0.901 ) but the fuzzy one has a **much better ROC AUC** ( 0.973 against 0.934 ): graded memberships give graded scores, while crisp terms produce many ties
* **A fuzzy rule set can be huge** ( 319 rules here, 1,900 with 3 conditions ): ranking by *weight × support* and keeping the best *k* shows that about **10 rules were as good as the whole set** ( more rules did not help ), and rules with a single condition were nearly as good as combinations ( 0.897 against 0.902 ); the differences among the variants are within the standard deviation of the folds
* The **linguistic terms are readable** like the rules of an expert system, but here the **partitions and the weights come from the data**; an expert can also write the partition and the rules by hand
* The partition is placed on training quantiles and **refitted inside every fold** of the cross-validation; the numbers of terms and conditions are a sensitivity analysis, not a tuning procedure
* With 171 test cases, differences smaller than the bootstrap intervals are not reliable